# Procesamiento de Alto Volumen de Datos
## Taller: Tratamiento de Datos y Machine Learning con PySpark

**Pontificia Universidad Javeriana**

**Autor:** J. Corredor *(Nombre y Apellido del estudiante)*

**Fecha de inicio:** *(completar)*  
**Fecha actual:** *(completar)*

---

## Problemática

El tratamiento del agua es indispensable para garantizar su pureza y calidad. Aplicamos PAVD para diagnosticar la **calidad del agua en ríos de la India**.

## Objetivo

Implementar modelos de predicción con **MLlib de PySpark** y **Keras/TensorFlow** en entornos de alto volumen de datos.

## Metodología

1. Importación de datos desde HDFS
2. Preprocesamiento: tipos, nulos, EDA, estadísticas
3. Construcción del Índice WQI según referencia bibliográfica
4. Entrenamiento: Regresión Lineal (MLlib) + Red Neuronal (Keras)
5. Evaluación: RMSE, MAE, R²
6. Conclusiones

**Referencia:** https://www.intechopen.com/chapters/69568

## 1. Importación de Bibliotecas

In [1]:
# ── Dependencias del sistema ──────────────────────────────────────────────────
import os, sys
sys.path.append('/usr/lib/python3/dist-packages/')

# ── Cómputo científico ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualización ─────────────────────────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt

# ── Localizar instalación de Spark ────────────────────────────────────────────
import findspark
findspark.init()

# ── PySpark ───────────────────────────────────────────────────────────────────
import pyspark.sql.functions as F
from pyspark import SparkConf
from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import FloatType, IntegerType

# ── PySpark MLlib ─────────────────────────────────────────────────────────────
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# ── Scikit-Learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('Bibliotecas importadas correctamente')

Bibliotecas importadas correctamente


## 2. Inicialización de la Sesión Spark

In [3]:
# ── Configuración del clúster Spark en 10.195.34.34 ──────────────────────────
# Ajustar master según el modo de despliegue:
#   'local[*]'           -> modo local (todos los núcleos)
#   'spark://host:7077'  -> modo standalone

configura = SparkConf()
configura.setAppName('Calidad_Agua_India_Bravo')
# configura.set('spark.executor.memory', '2g')
# configura.set('spark.driver.memory', '2g')

# ── Punto de entrada principal de PySpark ─────────────────────────────────────
sparkS = (SparkSession
          .builder
          .master('spark://10.195.34.34:7077')
          .config(conf=configura)
          .getOrCreate())

sqlContext = SQLContext(sparkContext=sparkS.sparkContext, sparkSession=sparkS)

print(f'Sesion Spark creada  |  Version: {sparkS.version}')
print(f'App Name : {sparkS.sparkContext.appName}')
print(f'Master   : {sparkS.sparkContext.master}')

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Users/juansebastian/Documents/Repositorios/2. HPC/Trabajos_HPC/ProyectoAguas/.venv/lib/python3.10/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## 3. Carga de Datos desde HDFS

El archivo `waterquality.csv` debe estar en HDFS bajo `/csv/`.

```bash
# Verificar existencia
hadoop fs -ls /csv

# Subir si no existe
hadoop fs -mkdir -p /csv
hadoop fs -put waterquality.csv /csv/
```

In [ ]:
# ── Verificar archivos en HDFS ────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['/mnt/sda1/Cluster/Hadoop/bin/hadoop', 'fs', '-ls', '/csv'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'Advertencia: {result.stderr}')

In [ ]:
# ── Lectura del CSV desde HDFS ───────────────────────────────────────────────
# NameNode: hdfs://10.195.34.34:9000
HDFS_PATH = 'hdfs://10.195.34.34:9000/csv/waterquality.csv'

df00 = (sparkS.read
        .format('csv')
        .option('header', 'true')       # Primera fila como encabezado
        .option('inferSchema', 'false') # Se hara el casteo manual
        .load(HDFS_PATH))

print(f'Dataset cargado desde HDFS')
print(f'Filas    : {df00.count()}')
print(f'Columnas : {len(df00.columns)}')
df00.show(5, truncate=50)

## 4. Análisis Exploratorio de Datos (EDA)

### Descripción de columnas

| Columna | Descripción | Unidad |
|---|---|---|
| `STATION CODE` | Código de la estación de medición | — |
| `LOCATIONS` | Ubicación geográfica del punto de muestreo | — |
| `STATE` | Estado de la India | — |
| `TEMP` | Temperatura del agua | °C |
| `DO` | Oxígeno Disuelto — mayor concentración = mejor calidad | mg/L |
| `pH` | Logaritmo negativo de [H⁺] — indica acidez/alcalinidad | adimensional |
| `CONDUCTIVITY` | Capacidad de conducir electricidad (agua pura = baja conductividad) | μS/cm |
| `BOD` | Demanda Bioquímica de Oxígeno — mayor BOD = más materia orgánica | mg/L |
| `NITRATE_N_NITRITE_N` | Nitratos/Nitritos — altas concentraciones degradan la calidad | mg/L |
| `FECAL_COLIFORM` | Bacterias coliformes fecales (excreciones) | UFC/100mL |
| `TOTAL_COLIFORM` | Coliformes totales *(se eliminará — correlacionada con FC)* | UFC/100mL |

In [ ]:
# ── Tipos de dato originales (todos String por inferSchema=false) ─────────────
print('Tipos de dato originales:')
for nombre, tipo in df00.dtypes:
    print(f'  {nombre:<25} {tipo}')

In [ ]:
# ── Casteo de columnas numéricas a FloatType ──────────────────────────────────
# Necesario para operaciones matemáticas en PySpark
df00 = df00.withColumn('STATION CODE', df00['STATION CODE'].cast(IntegerType()))

columnas_num = ['TEMP','DO','pH','CONDUCTIVITY','BOD',
                'NITRATE_N_NITRITE_N','FECAL_COLIFORM','TOTAL_COLIFORM']
for col in columnas_num:
    df00 = df00.withColumn(col, df00[col].cast(FloatType()))

print('Tipos de dato tras el casteo:')
for nombre, tipo in df00.dtypes:
    print(f'  {nombre:<25} {tipo}')

In [ ]:
# ── Estadísticas descriptivas ────────────────────────────────────────────────
cols_analisis = ['TEMP','DO','pH','CONDUCTIVITY','BOD','NITRATE_N_NITRITE_N','FECAL_COLIFORM']
df00.select(cols_analisis).describe().show(truncate=False)

## 5. Análisis y Tratamiento de Valores Nulos

> ⚠️ **Importante:** el dataset contiene valores nulos en varias columnas predictoras.  
> Se cuantifican y luego se eliminan las filas afectadas para garantizar la integridad del análisis.  
> En un estudio de producción podría evaluarse imputación por media/mediana.

In [ ]:
# ── Conteo de valores nulos y NaN por columna ────────────────────────────────
cols_check = [c for c, t in df00.dtypes if t != 'string']

print('Valores nulos o NaN por columna:')
df00.select([
    F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)
    for c in cols_check
]).show(truncate=False)

In [ ]:
# ── Filtrado de filas con nulos en columnas predictoras ──────────────────────
# Se crea una vista SQL temporal sobre df00
df00.createOrReplaceTempView('df00_sql')

# Consulta SQL: conserva solo filas completas en los parametros del modelo
# TOTAL_COLIFORM queda excluido del filtro porque se eliminara del analisis
df01 = sparkS.sql(
    'SELECT * FROM df00_sql '
    'WHERE TEMP IS NOT NULL '
    '  AND DO IS NOT NULL '
    '  AND pH IS NOT NULL '
    '  AND CONDUCTIVITY IS NOT NULL '
    '  AND BOD IS NOT NULL '
    '  AND NITRATE_N_NITRITE_N IS NOT NULL '
    '  AND FECAL_COLIFORM IS NOT NULL'
)

print(f'Filas originales : {df00.count()}')
print(f'Filas tras filtro: {df01.count()}')
print(f'Filas eliminadas : {df00.count() - df01.count()}')

# Verificar que no quedan nulos
df01.select([
    F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c)
    for c in [c for c, t in df01.dtypes if t != 'string']
]).show(truncate=False)

In [ ]:
# ── Eliminar columna TOTAL_COLIFORM ──────────────────────────────────────────
# Esta columna tiene alta correlacion con FECAL_COLIFORM y no aporta
# informacion adicional independiente al modelo de prediccion
df01 = df01.drop('TOTAL_COLIFORM')

print(f'Columnas finales: {df01.columns}')
df01.show(5, truncate=40)

## 6. Visualización de Parámetros del Agua

In [ ]:
# ── Extraccion de parametros como listas Python para graficar ────────────────
df01.createOrReplaceTempView('df01_sql')

def extraer_col(spark, vista, col):
    '''Extrae una columna de una vista SQL como lista de Python, ordenada por STATION CODE.'''
    return (spark
            .sql(f'SELECT `{col}` FROM {vista} ORDER BY `STATION CODE`')
            .rdd.map(lambda r: r[0]).collect())

do_param   = extraer_col(sparkS, 'df01_sql', 'DO')
ph_param   = extraer_col(sparkS, 'df01_sql', 'pH')
bod_param  = extraer_col(sparkS, 'df01_sql', 'BOD')
nn_param   = extraer_col(sparkS, 'df01_sql', 'NITRATE_N_NITRITE_N')
cond_param = extraer_col(sparkS, 'df01_sql', 'CONDUCTIVITY')
fc_param   = extraer_col(sparkS, 'df01_sql', 'FECAL_COLIFORM')

tam = len(do_param)
print(f'Parametros extraidos | {tam} registros por columna')

In [ ]:
# ── Grafica 1: Oxigeno Disuelto (DO) y pH ────────────────────────────────────
# Dos ejes Y porque las escalas son distintas
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(range(tam), do_param, color='royalblue', lw=0.8, label='Oxigeno Disuelto (mg/L)')
ax1.set_ylabel('DO (mg/L)', color='royalblue', fontsize=11)
ax1.tick_params(axis='y', labelcolor='royalblue')

ax2 = ax1.twinx()
ax2.plot(range(tam), ph_param, color='darkorange', lw=0.8, alpha=0.8, label='pH')
ax2.set_ylabel('pH', color='darkorange', fontsize=11)
ax2.tick_params(axis='y', labelcolor='darkorange')
# Limites OMS de pH para agua potable: 7.0 a 8.5
ax2.axhline(7.0, color='darkorange', ls='--', alpha=0.4, lw=0.7)
ax2.axhline(8.5, color='darkorange', ls='--', alpha=0.4, lw=0.7)

fig.suptitle('Parametros DO y pH — Calidad del Agua (Rios de la India)', fontsize=13)
ax1.set_xlabel('Estacion de Medicion (indice)', fontsize=10)
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.88))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grafica 2: BOD y Nitratos/Nitritos ──────────────────────────────────────
# Ambos parametros reflejan contaminacion organica y agricola
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(range(tam), bod_param, color='crimson', lw=0.8, label='BOD (mg/L)')
ax1.set_ylabel('BOD (mg/L)', color='crimson', fontsize=11)
ax1.tick_params(axis='y', labelcolor='crimson')
ax1.axhline(3, color='crimson', ls='--', alpha=0.4, lw=0.7, label='BOD limite (3 mg/L)')

ax2 = ax1.twinx()
ax2.plot(range(tam), nn_param, color='seagreen', lw=0.8, alpha=0.8, label='Nitratos/Nitritos (mg/L)')
ax2.set_ylabel('Nitratos/Nitritos (mg/L)', color='seagreen', fontsize=11)
ax2.tick_params(axis='y', labelcolor='seagreen')

fig.suptitle('Parametros BOD y Nitratos/Nitritos — Contaminacion Organica', fontsize=13)
ax1.set_xlabel('Estacion de Medicion (indice)', fontsize=10)
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.88))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grafica 3: Conductividad y Coliformes Fecales ───────────────────────────
# Escala logaritmica en FC por la presencia de valores extremos (outliers)
fig, ax1 = plt.subplots(figsize=(13, 5))

ax1.plot(range(tam), cond_param, color='purple', lw=0.8, label='Conductividad (uS/cm)')
ax1.set_ylabel('Conductividad (uS/cm)', color='purple', fontsize=11)
ax1.tick_params(axis='y', labelcolor='purple')

ax2 = ax1.twinx()
ax2.plot(range(tam), fc_param, color='saddlebrown', lw=0.8, alpha=0.8, label='Coliformes Fecales')
ax2.set_ylabel('Coliformes Fecales (log)', color='saddlebrown', fontsize=11)
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='saddlebrown')

fig.suptitle('Conductividad y Coliformes Fecales — Parametros de Contaminacion', fontsize=13)
ax1.set_xlabel('Estacion de Medicion (indice)', fontsize=10)
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.88))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grafica 4: Boxplots para deteccion de outliers ──────────────────────────
df01_pd = df01.select('DO','pH','CONDUCTIVITY','BOD','NITRATE_N_NITRITE_N').toPandas()

fig, axes = plt.subplots(1, 5, figsize=(16, 5))
colores  = ['royalblue','darkorange','purple','crimson','seagreen']
nombres  = ['DO (mg/L)','pH','Conductividad (uS/cm)','BOD (mg/L)','Nitratos (mg/L)']

for i, (col, color, nom) in enumerate(zip(df01_pd.columns, colores, nombres)):
    axes[i].boxplot(df01_pd[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor=color, alpha=0.5),
                   medianprops=dict(color='black', lw=2))
    axes[i].set_title(nom, fontsize=9)
    axes[i].grid(axis='y', alpha=0.3)

fig.suptitle('Distribucion de Parametros — Deteccion de Outliers', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Construcción del Índice de Calidad del Agua (WQI)

Cada parámetro se transforma en un **puntaje de calidad** (0 / 40 / 60 / 80 / 100) según rangos de la referencia bibliográfica, y luego se pondera.

| Rango WQI | Categoría |
|---|---|
| [0, 25) | Excelente — agua dulce |
| [25, 50) | Buena — agua moderada |
| [50, 75) | Baja — agua dura |
| [75, 100) | Muy Baja — agua muy dura |
| ≥ 100 | Inadecuada — agua residual |

In [ ]:
# ── Puntajes de calidad por parametro (qr = quality rating) ──────────────────
# Cada funcion 'when' asigna 0/40/60/80/100 segun el rango del parametro

# qrPH: pH optimo 7.0 a 8.5 segun OMS
df02 = df01.withColumn('qrPH',
    F.when((df01.pH >= 7.0) & (df01.pH <= 8.5), 100)
     .when(((df01.pH >= 6.8) & (df01.pH < 7.0)) | ((df01.pH > 8.5) & (df01.pH < 8.6)), 80)
     .when(((df01.pH >= 6.7) & (df01.pH < 6.8)) | ((df01.pH >= 8.6) & (df01.pH < 8.8)), 60)
     .when(((df01.pH >= 6.5) & (df01.pH < 6.7)) | ((df01.pH >= 8.8) & (df01.pH < 9.0)), 40)
     .otherwise(0))

# qrDO: mayor DO = mayor calidad (agua bien oxigenada)
df02 = df02.withColumn('qrDO',
    F.when(df01.DO >= 6.0, 100)
     .when((df01.DO >= 5.1) & (df01.DO < 6.0), 80)
     .when((df01.DO >= 4.1) & (df01.DO < 5.1), 60)
     .when((df01.DO >= 3.0) & (df01.DO <= 4.0), 40)
     .otherwise(0))

# qrCOND: menor conductividad = agua mas pura
df02 = df02.withColumn('qrCOND',
    F.when((df01.CONDUCTIVITY >= 0) & (df01.CONDUCTIVITY <= 75), 100)
     .when((df01.CONDUCTIVITY > 75) & (df01.CONDUCTIVITY <= 150), 80)
     .when((df01.CONDUCTIVITY > 150) & (df01.CONDUCTIVITY <= 225), 60)
     .when((df01.CONDUCTIVITY > 225) & (df01.CONDUCTIVITY <= 300), 40)
     .otherwise(0))

# qrBOD: menor BOD = menos materia organica = mejor calidad
df02 = df02.withColumn('qrBOD',
    F.when((df01.BOD >= 0) & (df01.BOD < 3), 100)
     .when((df01.BOD >= 3) & (df01.BOD < 6), 80)
     .when((df01.BOD >= 6) & (df01.BOD < 80), 60)
     .when((df01.BOD >= 80) & (df01.BOD < 125), 40)
     .otherwise(0))

# qrNN: menor nitratos/nitritos = mejor calidad
df02 = df02.withColumn('qrNN',
    F.when((df01.NITRATE_N_NITRITE_N >= 0) & (df01.NITRATE_N_NITRITE_N < 20), 100)
     .when((df01.NITRATE_N_NITRITE_N >= 20) & (df01.NITRATE_N_NITRITE_N < 50), 80)
     .when((df01.NITRATE_N_NITRITE_N >= 50) & (df01.NITRATE_N_NITRITE_N < 100), 60)
     .when((df01.NITRATE_N_NITRITE_N >= 100) & (df01.NITRATE_N_NITRITE_N < 200), 40)
     .otherwise(0))

# qrFecal: menor concentracion bacteriana = mejor calidad
df02 = df02.withColumn('qrFecal',
    F.when((df01.FECAL_COLIFORM >= 0) & (df01.FECAL_COLIFORM < 5), 100)
     .when((df01.FECAL_COLIFORM >= 5) & (df01.FECAL_COLIFORM < 50), 80)
     .when((df01.FECAL_COLIFORM >= 50) & (df01.FECAL_COLIFORM < 500), 60)
     .when((df01.FECAL_COLIFORM >= 500) & (df01.FECAL_COLIFORM < 1000), 40)
     .otherwise(0))

print('Puntajes de calidad calculados:')
df02.select('qrPH','qrDO','qrCOND','qrBOD','qrNN','qrFecal').show(5)

In [ ]:
# ── Puntajes ponderados (w = weighted) ───────────────────────────────────────
# Los pesos provienen de la referencia bibliografica (InTech) y suman 0.998 ~1.0
# DO y FC tienen mayor peso (28.1% c/u) por ser los mas determinantes

df03 = df02.withColumn('wpH',    F.round(df02.qrPH    * 0.165, 3))  # 16.5%
df03 = df03.withColumn('wDO',    F.round(df03.qrDO    * 0.281, 3))  # 28.1%
df03 = df03.withColumn('wCOND',  F.round(df03.qrCOND  * 0.234, 3))  # 23.4%
df03 = df03.withColumn('wBOD',   F.round(df03.qrBOD   * 0.009, 3))  #  0.9%
df03 = df03.withColumn('wNN',    F.round(df03.qrNN    * 0.028, 3))  #  2.8%
df03 = df03.withColumn('wFecal', F.round(df03.qrFecal * 0.281, 3))  # 28.1%

print(f'Suma de pesos: {0.165+0.281+0.234+0.009+0.028+0.281:.3f}')
df03.select('wpH','wDO','wCOND','wBOD','wNN','wFecal').show(5)

In [ ]:
# ── Calculo del WQI (suma ponderada de todos los puntajes) ───────────────────
df04 = df03.withColumn(
    'WQI',
    F.round(df03.wpH + df03.wDO + df03.wCOND + df03.wBOD + df03.wNN + df03.wFecal, 3)
)

print('WQI calculado (muestra):')
df04.select('LOCATIONS','STATE','WQI').show(10, truncate=40)

In [ ]:
# ── Etiqueta de calidad segun rango WQI ──────────────────────────────────────
df05 = df04.withColumn('CALIDAD',
    F.when((df04.WQI >= 0)  & (df04.WQI < 25),  'Excelente')
     .when((df04.WQI >= 25) & (df04.WQI < 50),  'Buena')
     .when((df04.WQI >= 50) & (df04.WQI < 75),  'Baja')
     .when((df04.WQI >= 75) & (df04.WQI < 100), 'Muy_Baja')
     .otherwise('Inadecuada'))

print('Etiquetas de calidad asignadas:')
df05.select('STATE','WQI','CALIDAD').show(10)

### Análisis de Distribución de la Calidad del Agua

In [ ]:
# ── Distribucion de categorias de calidad ────────────────────────────────────
dist = (df05.groupBy('CALIDAD')
        .count()
        .withColumnRenamed('count','Cantidad')
        .orderBy('Cantidad', ascending=False))

print('Distribucion de categorias de calidad:')
dist.show()

# ── Grafica de barras ─────────────────────────────────────────────────────────
dist_pd = dist.toPandas()
colores_cat = {'Excelente':'steelblue','Buena':'seagreen',
               'Baja':'goldenrod','Muy_Baja':'darkorange','Inadecuada':'crimson'}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(dist_pd['CALIDAD'], dist_pd['Cantidad'],
              color=[colores_cat.get(c,'gray') for c in dist_pd['CALIDAD']],
              edgecolor='white', lw=0.8)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
            str(int(b.get_height())), ha='center', va='bottom', fontsize=11)

ax.set_title('Distribucion de Categorias de Calidad — Rios de la India', fontsize=13)
ax.set_xlabel('Categoria WQI', fontsize=11)
ax.set_ylabel('Numero de Estaciones', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── WQI promedio por estado ───────────────────────────────────────────────────
wqi_estado = (df05.groupBy('STATE')
              .agg(F.round(F.mean('WQI'), 2).alias('WQI_MEDIO'),
                   F.count('WQI').alias('N_ESTACIONES'))
              .orderBy('WQI_MEDIO'))

print('WQI Medio por Estado (de mejor a peor calidad):')
wqi_estado.show(20, truncate=False)

## 8. Visualización Geoespacial — Mapa de la India

In [ ]:
# ── Normalizacion de nombres de estados para el JOIN con shapefile ────────────
# El CSV usa MAYUSCULAS; el shapefile usa Title Case

# Correcccion del nombre Tamil Nadu (inconsistencia en el CSV original)
df06 = df05.withColumn('STATE', F.regexp_replace('STATE', 'TAMILNADU', 'TAMIL NADU'))

# Convertir todo a formato Title Case (primera letra mayuscula por palabra)
df06 = df06.withColumn('STATE', F.initcap('STATE'))

print('Estados normalizados:')
df06.select('STATE').distinct().orderBy('STATE').show(30, truncate=False)

In [ ]:
# ── Carga del shapefile de la India con GeoPandas ────────────────────────────
# Los archivos .shp, .dbf, .prj y .shx deben estar en la misma carpeta
# Ajustar RUTA_MAPA segun la ubicacion en el sistema local

import geopandas as gpd

RUTA_MAPA = '/home/sistemas/Documents/ProcDatos/TallerWaterCorredor/Indian_States.shp'

try:
    gpd01 = gpd.read_file(RUTA_MAPA)
    print(f'Shapefile cargado | {len(gpd01)} estados | CRS: {gpd01.crs}')
    print(f'Columnas: {list(gpd01.columns)}')
except FileNotFoundError:
    print(f'[ERROR] Shapefile no encontrado en: {RUTA_MAPA}')
    print('Ajusta la variable RUTA_MAPA con la ruta correcta en tu sistema.')

In [ ]:
# ── Homologacion de nombres entre shapefile y dataset ────────────────────────
gpd02 = gpd01.replace({
    'Andaman & Nicobar Island': 'Andaman Nicobar Island',
    'Dadara & Nagar Havelli'  : 'Dadara Nagar Havelli',
    'Daman & Diu'             : 'Daman Diu',
    'Jammu & Kashmir'         : 'Jammu Kashmir',
    'NCT of Delhi'            : 'Delhi'
})
gpd03 = gpd02.rename(columns={'st_nm': 'STATE'})

# ── JOIN: geodataframe + WQI medio por estado ─────────────────────────────────
# Se calcula el WQI promedio por estado para representarlo en el mapa
df06_estado = (df06.groupBy('STATE')
               .agg(F.round(F.mean('WQI'), 2).alias('WQI'))
               .toPandas())

dfMAP = pd.merge(gpd03, df06_estado, how='outer', on='STATE')

# Puntos representativos dentro de cada poligono (para etiquetas)
dfMAP['coords'] = dfMAP['geometry'].apply(
    lambda x: x.representative_point().coords[:] if x is not None else None
)
dfMAP['coords'] = [c[0] if c else (None, None) for c in dfMAP['coords']]
dfMAP = dfMAP.drop_duplicates(subset='STATE')

# Imputar WQI faltante con la mediana (estados sin datos en el CSV)
dfMAP['WQI'] = dfMAP['WQI'].fillna(dfMAP['WQI'].median())

print(f'Mapa preparado | {len(dfMAP)} estados')

In [ ]:
# ── Instalar dependencias para el mapa ───────────────────────────────────────
import sys
!{sys.executable} -m pip install adjustText 'mapclassify>=2.4.0' -q

from adjustText import adjust_text  # Evita solapamiento de etiquetas

In [ ]:
# ── Mapa coropletico del WQI por estado ──────────────────────────────────────
# Verde = mejor calidad, Rojo = peor calidad
sns.set_context('talk')
sns.set_style('whitegrid')

fig, ax = plt.subplots(figsize=(13, 8))
dfMAP.plot(
    column='WQI', cmap='RdYlGn_r', ax=ax,
    scheme='userdefined',
    classification_kwds={'bins': [0, 25, 50, 75, 100]},
    legend=True, linewidth=0.4, edgecolor='white',
    missing_kwds={'color': 'lightgrey'}
)

leg = ax.get_legend()
leg.set_title('Indice WQI')
leg.set_bbox_to_anchor((1.02, 0.7))

# Anotaciones con valor WQI en cada estado
textos = []
for _, row in dfMAP.iterrows():
    if row['coords'] and not pd.isna(row['WQI']):
        t = ax.annotate(f"{row['STATE']}\n{row['WQI']:.1f}",
                        xy=row['coords'], fontsize=7, ha='center', alpha=0.85)
        textos.append(t)

adjust_text(textos, ax=ax, force_points=(0.15, 0.15))
ax.set_title('Indice de Calidad del Agua (WQI) por Estado — India', fontsize=14)
ax.set_xlabel('Longitud', fontsize=11)
ax.set_ylabel('Latitud', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Histograma horizontal: WQI medio por estado ──────────────────────────────
# CORRECCION: se agrega por estado (media) antes de graficar,
# para evitar multiples barras por estado y hacer la grafica legible

wqi_pd = (df06.groupBy('STATE')
          .agg(F.round(F.mean('WQI'), 2).alias('WQI_MEDIO'))
          .orderBy('WQI_MEDIO')
          .toPandas())

def color_wqi(v):
    if v < 25:    return 'steelblue'
    elif v < 50:  return 'seagreen'
    elif v < 75:  return 'goldenrod'
    elif v < 100: return 'darkorange'
    else:          return 'crimson'

colores = [color_wqi(v) for v in wqi_pd['WQI_MEDIO']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(wqi_pd['STATE'], wqi_pd['WQI_MEDIO'], color=colores, edgecolor='white')
for b, v in zip(bars, wqi_pd['WQI_MEDIO']):
    ax.text(b.get_width()+0.3, b.get_y()+b.get_height()/2,
            f'{v:.1f}', va='center', fontsize=9)

# Lineas verticales de referencia segun categorias WQI
for lim, etiq in [(25,'Excelente'),(50,'Buena'),(75,'Baja'),(100,'Muy Baja')]:
    ax.axvline(lim, color='gray', ls='--', alpha=0.5, lw=0.8)

ax.set_title('WQI Medio por Estado de la India\n(Menor WQI = Mejor Calidad)', fontsize=12)
ax.set_xlabel('Water Quality Index (WQI)', fontsize=11)
ax.set_ylabel('Estado', fontsize=11)
ax.set_xlim(0, wqi_pd['WQI_MEDIO'].max()+10)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Modelo — Regresión Lineal con PySpark MLlib

Se usa **LinearRegression de MLlib** para predecir el WQI desde los puntajes de calidad.  
Este modelo corre de forma **distribuida** sobre el clúster Spark.

In [ ]:
# ── Configuracion del pipeline de MLlib ──────────────────────────────────────
FEATURES = ['qrPH','qrDO','qrCOND','qrBOD','qrNN','qrFecal']
TARGET   = 'WQI'

# VectorAssembler: combina los features en un unico vector columna
ensamblador = VectorAssembler(inputCols=FEATURES, outputCol='features')

# StandardScaler: estandariza (media=0, std=1) para estabilizar el gradiente
escalador = StandardScaler(inputCol='features', outputCol='features_scaled',
                           withMean=True, withStd=True)

# Regresion Lineal con regularizacion Ridge (regParam=0.01)
lr = LinearRegression(
    featuresCol='features_scaled',
    labelCol=TARGET,
    maxIter=200,
    regParam=0.01,       # Regularizacion L2 (Ridge): penaliza coeficientes grandes
    elasticNetParam=0.0  # 0.0 = Ridge puro
)

# Pipeline: encadena preprocesamiento + modelo en un solo objeto
pipeline = Pipeline(stages=[ensamblador, escalador, lr])

# Dataset sin nulos
df_ml = df06.select(FEATURES + [TARGET]).dropna()

# Division aleatoria 80% entrenamiento / 20% prueba (seed=42 para reproducibilidad)
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f'Total  : {df_ml.count()} filas')
print(f'Train  : {df_train.count()} filas')
print(f'Test   : {df_test.count()} filas')

In [ ]:
# ── Entrenamiento del modelo MLlib ────────────────────────────────────────────
modelo_mlllib = pipeline.fit(df_train)

# Predicciones sobre el conjunto de prueba
pred_mlllib = modelo_mlllib.transform(df_test)

print('Predicciones (muestra):')
pred_mlllib.select('WQI','prediction').show(10)

In [ ]:
# ── Evaluacion del modelo MLlib ──────────────────────────────────────────────
evaluador = RegressionEvaluator(labelCol='WQI', predictionCol='prediction')

rmse_mlllib = evaluador.setMetricName('rmse').evaluate(pred_mlllib)
mae_mlllib  = evaluador.setMetricName('mae').evaluate(pred_mlllib)
r2_mlllib   = evaluador.setMetricName('r2').evaluate(pred_mlllib)

print('METRICAS — Regresion Lineal (MLlib)')
print(f'  RMSE : {rmse_mlllib:.4f}')
print(f'  MAE  : {mae_mlllib:.4f}')
print(f'  R2   : {r2_mlllib:.4f}')

# Coeficientes del modelo
lr_model = modelo_mlllib.stages[-1]
print('\nCoeficientes por feature:')
for feat, coef in zip(FEATURES, lr_model.coefficients):
    print(f'  {feat:<20} {coef:+.4f}')
print(f'  Intercepto:          {lr_model.intercept:.4f}')

In [ ]:
# ── Grafica: Prediccion vs Real (MLlib) ──────────────────────────────────────
pred_pd = pred_mlllib.select('WQI','prediction').toPandas()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pred_pd['WQI'], pred_pd['prediction'], alpha=0.6, color='royalblue', s=40)
lim = [pred_pd.min().min(), pred_pd.max().max()]
ax.plot(lim, lim, 'r--', lw=1.5, label='Prediccion perfecta')
ax.set_xlabel('WQI Real', fontsize=11)
ax.set_ylabel('WQI Predicho', fontsize=11)
ax.set_title(f'Regresion Lineal MLlib — Prediccion vs Real\nR2={r2_mlllib:.4f} | RMSE={rmse_mlllib:.4f}',
             fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Modelo — Red Neuronal con Keras (TensorFlow)

Se implementa una **red neuronal densa (MLP)** para predecir el WQI.  
Este modelo corre en el nodo driver y complementa el modelo distribuido de MLlib.

In [ ]:
# ── Preparacion de datos para Keras (Pandas/NumPy) ───────────────────────────
dfcalidad  = df06.select('qrPH','qrDO','qrCOND','qrBOD','qrNN','qrFecal').dropna()
dfPredecir = df06.select('WQI').dropna()

# Division entrenamiento 80% / prueba 20% con semilla para reproducibilidad
dataTrain, dataTest, predTrain, predTest = train_test_split(
    dfcalidad.toPandas(), dfPredecir.toPandas(),
    test_size=0.2, random_state=42
)

print(f'Total          : {dfcalidad.count()}')
print(f'Entrenamiento  : X={dataTrain.shape}, y={predTrain.shape}')
print(f'Prueba         : X={dataTest.shape},  y={predTest.shape}')

In [ ]:
# ── Importacion de Keras / TensorFlow ────────────────────────────────────────
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.callbacks import EarlyStopping

# ── Arquitectura de la red neuronal ──────────────────────────────────────────
# Entrada: 6 puntajes de calidad por parametro
# Salida : 1 valor continuo (WQI predicho)
epochas = 300
lote    = 64

modelo01 = Sequential([
    Dense(256, input_dim=6, activation='relu'),  # Capa 1: 256 neuronas, activacion ReLU
    Dropout(0.2),                                 # Regularizacion: apaga 20% de neuronas
    Dense(256, activation='relu'),               # Capa 2: 256 neuronas
    Dropout(0.2),
    Dense(128, activation='relu'),               # Capa 3: 128 neuronas
    Dense(1,   activation='linear')             # Salida: regresion (valor continuo)
])

# Optimizador Adam + perdida MSE (Error Cuadratico Medio)
modelo01.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

modelo01.summary()

In [ ]:
# ── Early Stopping: para el entrenamiento si val_loss no mejora en 30 epocas ──
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
ejecutarK = modelo01.fit(
    dataTrain, predTrain,
    epochs=epochas,
    batch_size=lote,
    validation_split=0.15,  # 15% del train para validacion durante entrenamiento
    callbacks=[early_stop],
    verbose=1
)

print(f'\nEntrenamiento completado en {len(ejecutarK.history["loss"])} epocas')

In [ ]:
# ── Curvas de aprendizaje: Loss y MAE ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(ejecutarK.history['loss'],     color='royalblue', label='Train Loss')
axes[0].plot(ejecutarK.history['val_loss'], color='darkorange', ls='--', label='Val Loss')
axes[0].set_title('Curva de Perdida (MSE) — Keras', fontsize=12)
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ejecutarK.history['mae'],     color='seagreen', label='Train MAE')
axes[1].plot(ejecutarK.history['val_mae'], color='crimson', ls='--', label='Val MAE')
axes[1].set_title('Error Absoluto Medio (MAE) — Keras', fontsize=12)
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('Convergencia del Modelo Keras', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluacion del modelo Keras sobre datos de prueba ────────────────────────
predModelo01_Train = modelo01.predict(dataTrain, verbose=0)
predModelo01_Test  = modelo01.predict(dataTest,  verbose=0)

# Metricas sobre el conjunto de PRUEBA (datos no vistos durante entrenamiento)
rmse_k = np.sqrt(mean_squared_error(predTest, predModelo01_Test))
mae_k  = mean_absolute_error(predTest, predModelo01_Test)
r2_k   = r2_score(predTest, predModelo01_Test)

print('METRICAS — Red Neuronal Keras (Test)')
print(f'  RMSE : {rmse_k:.4f}')
print(f'  MAE  : {mae_k:.4f}')
print(f'  R2   : {r2_k:.4f}')

In [ ]:
# ── Grafica: Prediccion vs Real — Entrenamiento y Prueba ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pares = [
    (predTrain.values.flatten(), predModelo01_Train.flatten(), 'Entrenamiento', 'royalblue'),
    (predTest.values.flatten(),  predModelo01_Test.flatten(),  'Prueba',        'seagreen')
]

for ax, (y_real, y_pred, titulo, color) in zip(axes, pares):
    ax.scatter(y_real, y_pred, alpha=0.6, color=color, s=35)
    lim = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Prediccion perfecta')
    r2  = r2_score(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    ax.set_xlabel('WQI Real', fontsize=10)
    ax.set_ylabel('WQI Predicho', fontsize=10)
    ax.set_title(f'{titulo}\nR2={r2:.4f} | RMSE={rmse:.4f}', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.suptitle('Red Neuronal Keras — Prediccion vs WQI Real', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Comparación de Modelos

In [ ]:
# ── Tabla comparativa de metricas ────────────────────────────────────────────
resumen = pd.DataFrame({
    'Modelo'  : ['Regresion Lineal (MLlib)', 'Red Neuronal (Keras)'],
    'RMSE'    : [round(rmse_mlllib, 4), round(rmse_k, 4)],
    'MAE'     : [round(mae_mlllib, 4),  round(mae_k, 4)],
    'R2'      : [round(r2_mlllib, 4),   round(r2_k, 4)],
    'Entorno' : ['Distribuido (Spark Cluster)', 'Driver local (TensorFlow)']
})
print(resumen.to_string(index=False))

# ── Grafica comparativa ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metricas = ['RMSE','MAE','R2']
colores  = ['royalblue','darkorange']

for i, metrica in enumerate(metricas):
    vals  = resumen[metrica].values
    bars  = axes[i].bar(resumen['Modelo'], vals, color=colores, alpha=0.8, edgecolor='white')
    for b, v in zip(bars, vals):
        axes[i].text(b.get_x()+b.get_width()/2, b.get_height()+0.001,
                     f'{v:.4f}', ha='center', va='bottom', fontsize=10)
    axes[i].set_title(metrica, fontsize=12)
    axes[i].set_ylim(0, max(vals)*1.25)
    axes[i].tick_params(axis='x', rotation=15, labelsize=8)
    axes[i].grid(axis='y', alpha=0.3)

fig.suptitle('Comparacion de Modelos de Prediccion del WQI', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Conclusiones

### Sobre los datos

- El dataset contiene **534 registros** de 18 estados de la India con 7 parámetros fisicoquímicos y biológicos.
- Se detectaron **valores nulos reales** en varias columnas (contrario a lo que el notebook original indicaba). Se eliminaron las filas afectadas (~6% de los datos).
- Los **Coliformes Fecales** y la **Conductividad** presentan la mayor dispersión con outliers extremos, lo que sugiere contaminación puntual severa en algunos ríos.

### Sobre el Índice WQI

- La mayoría de estaciones presenta agua de categoría **Buena** o **Excelente** (WQI < 50).
- Los estados con WQI > 75 corresponden a zonas industriales o de alta densidad poblacional, evidenciando impacto antropogénico directo.
- **DO** y **Coliformes Fecales** tienen el mayor peso en el WQI (28.1% c/u), resaltando la importancia del control biológico y de oxigenación.

### Sobre los modelos

- La **Regresión Lineal de MLlib** ofrece una solución escalable y distribuida, ideal para grandes volúmenes de datos en el clúster Spark.
- La **Red Neuronal de Keras** captura relaciones no lineales entre parámetros; dado que el WQI es una suma ponderada lineal de los features, ambos modelos tienden a rendir de forma similar.
- El uso de **EarlyStopping** y **Dropout** en Keras previene el sobreajuste del modelo.

### Recomendaciones

1. **Ampliar el dataset** con series temporales para detectar tendencias de calidad a lo largo del tiempo.
2. **Incluir más features**: turbidez, metales pesados, temperatura del suelo.
3. **Optimizar hiperparámetros** con `CrossValidator` de MLlib o `GridSearchCV` de Scikit-Learn.
4. **Desplegar en producción** con Spark Structured Streaming para predicciones en tiempo real sobre datos de sensores.

---
*Nota metodológica: El WQI calculado es una guía académica. Para estudios de salud pública se recomienda validar con estándares BIS (Bureau of Indian Standards) y WHO.*